# Session 8: Noise in chemical measurements

A drinking-water laboratory reports the lead in each home's tap water as a number, or as zero when the result is below the lab's method detection limit (MDL), which federal rules require to be at most 1 µg/L. The lab's manager sets how long the ICP-MS counts lead ions for each reading, and the MDL follows from that choice and from the lab's own blanks. This notebook finds which part of the noise a longer count removes and which part sets the MDL. Section 1 works through a real ICP-MS count trace and one lab's MDL study. Section 2 computes the MDL for a lab assigned to you, at four integration times, and reports its homes. Section 3 asks how the background is subtracted when it drifts during the day. Section 4 is the judgment cell.

The count trace in `data/ag_trace_1ms.csv` is real: one 60 s run of dissolved silver at 0.1 µg/L, read in 1 ms windows, from I. Abad Álvaro, E. Bolea Morales and F. Laborda García, "SP-ICP-MS Raw Data", Zenodo, 2026, [doi:10.5281/zenodo.18745872](https://doi.org/10.5281/zenodo.18745872), CC BY 4.0, accessed 2026-10-02, described in *Talanta* (2026) 129575, [doi:10.1016/j.talanta.2026.129575](https://doi.org/10.1016/j.talanta.2026.129575). The lead studies in `data/method_study.csv` and `data/labs/` are simulated from a stated counting model; `data/README.md` describes both.

**To submit.** Put your name in the next cell. Click **Restart** in the notebook toolbar, then **Run All**, check that every cell ran without an error, save, and upload `activity.ipynb` to the Session 8 assignment on Canvas by 1:30 pm on Thursday, October 29. Submissions are individual. Every lab's file is in the repository, so the graders' run reads the same data yours did. The notebook is scored 0 to 3: 1 if it runs from top to bottom after a restart, and 0 to 2 for the judgment cell, by the rubric in the syllabus, with the format in `ai-practices.md`.

*Name:*



## Setup and imports

The next cell imports `sys`, `Path` from `pathlib`, NumPy, pandas, Matplotlib and `linregress` from SciPy, which you used for calibration lines in session 2. New this session is NumPy's random number generator, made in section 1 from `RNG_SEED`: `rng.poisson(mean, size=n)` draws `n` counts from a Poisson distribution with that mean, the distribution that counting detected ions follows. The cell prints nothing when it works.

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import linregress

## Parameters

The count trace and the window lengths it is summed into, the seed for the simulation; the lead studies' four integration times; the Student's t value Appendix B uses for seven replicates, and the MDL the rule requires; then your lab, set in section 2. Leave `LAB` as `CHANGE-ME` until section 2.

In [ ]:
TRACE_CSV = Path("data/ag_trace_1ms.csv")
DWELL_S = 0.001                              # one reading of the trace, s
WINDOW_DWELLS = [1, 4, 16, 64, 256, 1024, 4096]  # readings summed per window: 1 ms to 4.1 s, each four times the last
RNG_SEED = 427

METHOD_CSV = Path("data/method_study.csv")
INTEGRATION_TIMES_S = [0.025, 0.1, 0.4, 1.6]  # s, each four times the last
T_99 = 3.143                                 # Student's t, 99%, 6 degrees of freedom: 40 CFR 136 Appendix B, seven replicates
REQUIRED_MDL_UG_L = 1.0                      # 40 CFR 141.89: a lead MDL of 0.001 mg/L
LAB = "CHANGE-ME"                            # your assigned lab, for example "lab_07", section 2

## 1. Explore: what a longer count buys

A detector that counts ions gives a whole number each time it is read, and the number scatters even when the solution does not change. If the ions arrive independently, the counts follow a Poisson distribution, whose variance equals its mean, so a reading of N counts carries a standard deviation of √N and a relative standard deviation of 1/√N. The first cell reads the real trace, one count per 1 ms reading under a one-line header, and checks that rule on the data: the variance over the mean should be close to 1.

In [ ]:
counts = np.loadtxt(TRACE_CSV, skiprows=1)   # the header line is "107Ag, counts": the isotope, then the unit
mean_counts = counts.mean()
print(f"{counts.size} readings of {DWELL_S * 1000:.0f} ms, {counts.size * DWELL_S:.1f} s in all")
print(f"mean {mean_counts:.3f} counts per reading, {mean_counts / DWELL_S:.0f} counts per second")
print(f"variance {counts.var(ddof=1):.3f}, variance over mean {counts.var(ddof=1) / mean_counts:.3f}")

A longer integration time is the same trace summed over longer windows. The next cell sums 1, 4, 16 and up to 4,096 readings at a time, so each window is four times the last, and prints each window's mean and relative standard deviation beside 1/√N. It also draws the same number of windows from a Poisson distribution with the same mean, with `rng.poisson`, to show what pure counting noise would give. The plot puts all three on log axes, where 1/√N is a straight line falling by half for each fourfold step.

In [ ]:
rng = np.random.default_rng(RNG_SEED)
rows = []
for n in WINDOW_DWELLS:
    n_windows = counts.size // n
    windows = counts[: n_windows * n].reshape(n_windows, n).sum(axis=1)   # each row is one window of n readings
    simulated = rng.poisson(windows.mean(), size=n_windows)
    rows.append({"window_s": n * DWELL_S, "windows": n_windows, "mean_counts": windows.mean(),
                 "relative_sd": windows.std(ddof=1) / windows.mean(),
                 "simulated_relative_sd": simulated.std(ddof=1) / simulated.mean(),
                 "one_over_root_n": 1 / np.sqrt(windows.mean())})
window_table = pd.DataFrame(rows)
print(window_table.round(4).to_string(index=False))

fig, ax = plt.subplots()
ax.plot(window_table["window_s"], window_table["one_over_root_n"], "-", color="gray", label="1/√N")
ax.plot(window_table["window_s"], window_table["simulated_relative_sd"], "s", color="black", fillstyle="none", label="Poisson simulation")
ax.plot(window_table["window_s"], window_table["relative_sd"], "o-", color="black", label="measured trace")
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("window (s)")
ax.set_ylabel("relative standard deviation of the counts")
ax.set_title("Dissolved Ag at 0.1 µg/L, 107Ag counts")
ax.legend()
fig.tight_layout()
plt.show()

The measured relative standard deviation falls from 0.612 at 1 ms to 0.166 at 16 ms, close to 1/√N (0.598 and 0.149) and to the Poisson simulation, and then falls more slowly, to 0.0247 at 1.02 s and 0.0164 at 4.10 s, where 1/√N gives 0.0187 and 0.0093. The variance over the mean of a single reading, 1.048, already showed counting noise at 1 ms; the extra noise past about 16 ms comes from the plasma and the sample introduction, and counting longer does not average it away.

**Explain.** In two sentences: from the table, by what factor did the step from 1.02 s to 4.10 s lower the trace's relative standard deviation, against the factor of 2 that 1/√N gives for every fourfold step? What does that tell you about counting four times longer at that window?

*Your answer:*

The lead studies come next. `data/method_study.csv` is one laboratory's MDL study by 40 CFR 136 Appendix B, simulated from the counting model in `data/README.md`: seven method blanks and seven spikes at 2 µg/L, prepared through the whole method and read on three days, each at four integration times. Each day opens with five calibration standards, so the next cell fits each day's calibration line at each time, counts per second against µg/L, and converts every reading to µg/L with it. The function is reused in section 2 for your lab.

In [ ]:
def to_concentration(readings):
    """Lead in every reading, from the day's calibration line at that integration time.

    Parameters: readings, a run as read from its CSV, one row per reading.
    Returns: a copy with rate_cps (counts per second) and conc_ug_L (lead, ug/L) added.
    """
    results = readings.copy()
    results["rate_cps"] = results["counts"] / results["integration_s"]   # counts per second
    lines = []
    for day in sorted(results["day"].unique()):
        for t in INTEGRATION_TIMES_S:
            standards = results[(results["day"] == day) & (results["integration_s"] == t) & (results["kind"] == "standard")]
            fit = linregress(standards["nominal_ug_L"], standards["rate_cps"])
            lines.append({"day": day, "integration_s": t, "slope_cps_per_ug_L": fit.slope, "intercept_cps": fit.intercept})
    calibration = pd.DataFrame(lines)
    results = results.merge(calibration, on=["day", "integration_s"])
    results["conc_ug_L"] = (results["rate_cps"] - results["intercept_cps"]) / results["slope_cps_per_ug_L"]
    return results


study = to_concentration(pd.read_csv(METHOD_CSV))
print(study.groupby("kind").size())
print(study.groupby("integration_s")["slope_cps_per_ug_L"].mean().round(0))

The spike MDL of Appendix B is t times the standard deviation of the seven spikes, MDL = t × s, where t = 3.143 is Student's t at 99% for six degrees of freedom. The next cell computes it at each integration time and plots it against time on a log axis, beside a line falling as 1/√t from the shortest time, which is what counting noise alone would give. The dashed line is the 1 µg/L the rule requires.

In [ ]:
spike_mdl = []
for t in INTEGRATION_TIMES_S:
    spikes = study.loc[(study["kind"] == "spike") & (study["integration_s"] == t), "conc_ug_L"]
    spike_mdl.append(T_99 * spikes.std())
times = np.array(INTEGRATION_TIMES_S)
counting_only = spike_mdl[0] * np.sqrt(times[0] / times)   # ug/L, falling as 1/sqrt(t)
for t, mdl in zip(INTEGRATION_TIMES_S, spike_mdl):
    print(f"{t:>6} s: spike MDL {mdl:.3f} ug/L")

fig, ax = plt.subplots()
ax.plot(times, counting_only, "-", color="gray", label="counting noise alone, 1/√t")
ax.plot(times, spike_mdl, "o-", color="black", label="spike MDL, method study")
ax.axhline(REQUIRED_MDL_UG_L, color="black", linestyle="--", linewidth=1, label="required MDL")
ax.set_xscale("log")
ax.set_xlabel("integration time (s)")
ax.set_ylim(bottom=0)
ax.set_ylabel("spike MDL (µg/L)")
ax.set_title("Method study, spike MDL against integration time")
ax.legend()
fig.tight_layout()
plt.show()

The spike MDL falls from 1.306 µg/L at 0.025 s to 0.757 at 0.1 s and 0.298 at 0.4 s, roughly as 1/√t, and then only to 0.281 at 1.6 s, where counting noise alone would give 0.163 µg/L. Past 0.4 s two things a longer count cannot change set the spikes' scatter: the lead each spike picked up from its reagents, and the 2% reading-to-reading noise the model carries for the plasma, the floor section 1's trace showed.

## 2. Apply: set your lab's integration time and report its homes

Your lab ran the same MDL study and also read 40 first-draw tap-water samples from homes, every one at all four integration times. Its reagents carry a different amount of lead and its background drifts at a different rate, so its MDL behaves differently from section 1's. Start a new conversation in Claude Code now: type `/` in the prompt box and choose **Clear conversation**. If it is not in the menu, open the Command Palette (**View**, then **Command Palette**) and run **Claude Code: Open in New Tab**.

**[TASK 1]** Set `LAB` in the Parameters cell to your assigned lab, run the Parameters cell, then run the next cell. Until `LAB` is set, it stops with `FileNotFoundError` for `data/labs/CHANGE-ME.csv`.

In [ ]:
lab_results = to_concentration(pd.read_csv(f"data/labs/{LAB}.csv"))
print(LAB)
print(lab_results[lab_results["integration_s"] == INTEGRATION_TIMES_S[0]].groupby("kind").size())

**Ask the assistant** before TASK 2, in Claude Code, with today's AI practice, *correct early, and clear context*. First save the notebook (**File**, then **Save**), so the assistant reads the `LAB` you set. Then ask the way you might without the notes open:

`What is the detection limit of my lab's blanks at each integration time? The readings are in lab_results in my notebook.`

Read its answer against notes section 3 before you use any of it. Does it use t = 3.143, as in `T_99`, or 3? Does it add the blanks' mean? Does it compute the spike MDL too, and report the greater? If all three hold, the answer is verified: note what you checked it against. If one fails, correct it at once, in one sentence that names the fix, for example `Use 40 CFR 136 Appendix B: the blanks' mean plus T_99 times their SD, and report the greater of that and the spike MDL.` If two corrections have not fixed it, stop correcting.

Either way, now clear the conversation (`/`, then **Clear conversation**, or **Claude Code: Open in New Tab**) and send a better first prompt, one that carries the notes' words and your notebook's names:

`Read sessions/S08-noise/notes.md, section 3, on the method detection limit. In my notebook, lab_results holds every reading of my lab, with kind ("standard", "cal_blank", "blank", "spike" or "home"), integration_s from INTEGRATION_TIMES_S, and conc_ug_L. For each integration time, compute the spike MDL and the blank MDL by 40 CFR 136 Appendix B and report the greater, in µg/L.`

Compare its answer with the first conversation's, and with the table TASK 2 builds next. What the assistant suggested and what you verified or corrected, with the value you checked it against, is one candidate for part (b) of the judgment cell.

**[TASK 2]** Make your lab's MDL table. The cell computes the spike MDL and 3 times the blanks' standard deviation at each integration time, and reports the greater of the spike and blank MDLs as the lab's MDL; the line for the blank MDL is left to you. Until it is filled, the cell stops with `TypeError: '<' not supported between instances of 'float' and 'ellipsis'`.

In [ ]:
rows = []
for t in INTEGRATION_TIMES_S:
    at_t = lab_results[lab_results["integration_s"] == t]
    spikes = at_t.loc[at_t["kind"] == "spike", "conc_ug_L"]
    blanks = at_t.loc[at_t["kind"] == "blank", "conc_ug_L"]
    mdl_spike_ug_L = T_99 * spikes.std()
    mdl_blank_ug_L = ...   # [TASK 2] the blank MDL of notes section 3 from blanks, in ug/L
    rows.append({"integration_s": t, "blank_mean_ug_L": blanks.mean(), "mdl_spike_ug_L": mdl_spike_ug_L, "mdl_blank_ug_L": mdl_blank_ug_L,
                 "mdl_ug_L": max(mdl_spike_ug_L, mdl_blank_ug_L), "three_sd_blanks_ug_L": 3 * blanks.std()})
mdl_table = pd.DataFrame(rows).set_index("integration_s")
mdl_table.round(3)

The next cell draws the table: each MDL against integration time on a log axis, with the 1 µg/L the rule requires dashed.

In [ ]:
fig, ax = plt.subplots()
ax.plot(mdl_table.index, mdl_table["mdl_spike_ug_L"], "s--", color="black", fillstyle="none", label="spike MDL")
ax.plot(mdl_table.index, mdl_table["mdl_blank_ug_L"], "^:", color="black", fillstyle="none", label="blank MDL")
ax.plot(mdl_table.index, mdl_table["mdl_ug_L"], "o-", color="black", label="MDL, the greater")
ax.axhline(REQUIRED_MDL_UG_L, color="gray", linestyle="--", linewidth=1, label="required MDL")
ax.set_xscale("log")
ax.set_xlabel("integration time (s)")
ax.set_ylim(bottom=0)
ax.set_ylabel("MDL (µg/L)")
ax.set_title(f"{LAB}, MDL against integration time")
ax.legend()
fig.tight_layout()
plt.show()

*Write your one sentence description of the plot here.*

**[TASK 3]** Choose the integration time your lab will use, from the table: one whose MDL meets the rule, and a reason you can state from the numbers, such as how much each fourfold step lowered the MDL against the halving that counting alone would give. Until it is set, the cell stops with `TypeError: float() argument must be a string or a real number, not 'ellipsis'`.

In [ ]:
INTEGRATION_S = ...   # [TASK 3] your choice, one of INTEGRATION_TIMES_S, in s
chosen = mdl_table.loc[float(INTEGRATION_S)]
print(f"{LAB} at {INTEGRATION_S} s: spike MDL {chosen['mdl_spike_ug_L']:.3f}, blank MDL {chosen['mdl_blank_ug_L']:.3f}, "
      f"MDL {chosen['mdl_ug_L']:.3f} ug/L; meets {REQUIRED_MDL_UG_L} ug/L: {chosen['mdl_ug_L'] <= REQUIRED_MDL_UG_L}")

**[TASK 4]** Report your lab's homes at the chosen time. A result below the MDL is reported as zero, which 40 CFR 141.89 requires; the rule lets a result between the MDL and 5 µg/L be reported as measured or as 2.5 µg/L, and this notebook reports it as measured. The `detected` column keeps which is which. The line that decides `detected` is left to you. Until it is filled, the cell stops at the line after it with `TypeError: unsupported operand type(s) for +: 'ellipsis' and 'ellipsis'`. The cell then counts how many calls would change if the MDL had been the spike MDL alone, or 3 times the blanks' standard deviation.

In [ ]:
homes = lab_results[(lab_results["kind"] == "home") & (lab_results["integration_s"] == INTEGRATION_S)].copy()
homes["detected"] = ...   # [TASK 4] True where the home is reported as a number, by chosen["mdl_ug_L"]
n_numbers = int(homes["detected"].sum())
homes["reported_ug_L"] = np.where(homes["detected"], homes["conc_ug_L"], 0.0)   # ug/L, zero below the MDL
print(f"{n_numbers} of {len(homes)} homes reported as a number, {len(homes) - n_numbers} as zero")
for label in ["mdl_spike_ug_L", "three_sd_blanks_ug_L"]:
    changed = int(((homes["conc_ug_L"] >= chosen[label]) != homes["detected"]).sum())
    print(f"with {label} as the limit ({chosen[label]:.3f} ug/L), calls that would change: {changed}")

**Explain.** Before you compare: in two sentences, say by what factor each fourfold step from 0.1 s lowered your lab's MDL, against the halving counting alone would give, and which MDL, spike or blank, sets it at your chosen time, citing your table, and how many of your homes are reported as numbers there. Then compare with a neighbor: set your labs, your MDLs at each time and your chosen times side by side. Add one sentence on where your two answers differ and what in your labs' blanks explains it.

*Your answer:*

## 3. Analyze: choose how the background is subtracted

Every reading so far had the day's calibration intercept subtracted, which is the background at the start of the day. The background rises through the run, so a blank read in the afternoon reads above zero, and that drift enters the blank MDL. The run also read a calibration blank, clean acid with no lead, before every fourth sample. Two treatments are defensible, and each has a cost. Treatment A, used so far, subtracts the day's opening background and needs no extra readings, but every later reading, homes included, carries the drift as apparent lead. Treatment B subtracts the calibration blanks' rate interpolated to each reading's minute, which follows the drift, but needs the calibration blanks the run read before every fourth sample and at the end of each day, where EPA Method 200.8 reads one for every ten samples. The cell recomputes everything under B and prints, for each treatment and time, the spike MDL, the blank MDL, the MDL, the calls the spike MDL alone would change and the homes reported as numbers; then the lead A adds to the mean of your homes at each time, and the calibration blanks the run read.

In [ ]:
bracketed = lab_results.copy()
for day in sorted(bracketed["day"].unique()):
    for t in INTEGRATION_TIMES_S:
        rows_now = (bracketed["day"] == day) & (bracketed["integration_s"] == t)
        cal = bracketed[rows_now & (bracketed["kind"] == "cal_blank")]
        background_cps = np.interp(bracketed.loc[rows_now, "minute"], cal["minute"], cal["rate_cps"])
        bracketed.loc[rows_now, "conc_ug_L"] = ((bracketed.loc[rows_now, "rate_cps"] - background_cps)
                                                / bracketed.loc[rows_now, "slope_cps_per_ug_L"])

compare = []
for label, results in [("A", lab_results), ("B", bracketed)]:
    for t in INTEGRATION_TIMES_S:
        at_t = results[results["integration_s"] == t]
        spikes = at_t.loc[at_t["kind"] == "spike", "conc_ug_L"]
        blanks = at_t.loc[at_t["kind"] == "blank", "conc_ug_L"]
        mdl_spike = T_99 * spikes.std()
        mdl_blank = max(blanks.mean(), 0) + T_99 * blanks.std()
        mdl = max(mdl_spike, mdl_blank)
        home_values = at_t.loc[at_t["kind"] == "home", "conc_ug_L"]
        compare.append({"treatment": label, "time_s": t, "spike_mdl": mdl_spike, "blank_mdl": mdl_blank, "mdl": mdl,
                        "meets_rule": mdl <= REQUIRED_MDL_UG_L, "homes_numbers": int((home_values >= mdl).sum()),
                        "changed_spike_only": int(((home_values >= mdl_spike) != (home_values >= mdl)).sum())})
compare = pd.DataFrame(compare)
print("MDLs in ug/L")
print(compare.round(3).to_string(index=False))

for t in INTEGRATION_TIMES_S:
    at_t = (lab_results["kind"] == "home") & (lab_results["integration_s"] == t)
    drift_added = (lab_results.loc[at_t, "conc_ug_L"] - bracketed.loc[at_t, "conc_ug_L"]).mean()   # ug/L
    print(f"at {t} s, A adds {drift_added:.3f} ug/L to the mean of your homes")
one_time = lab_results[lab_results["integration_s"] == INTEGRATION_TIMES_S[0]]
n_cal = int((one_time["kind"] == "cal_blank").sum())
n_samples = int(((one_time["kind"] == "blank") | (one_time["kind"] == "spike") | (one_time["kind"] == "home")).sum())
print(f"the run read {n_cal} calibration blanks for {n_samples} samples over three days")

**Explain.** For each treatment at your chosen time, give the spike MDL, the blank MDL, the MDL, the calls the spike MDL alone would change and the homes reported as numbers, and the shortest time that meets the rule. Then give the lead treatment A adds to your homes and the calibration blanks treatment B needs. This is the evidence part (a) of the judgment cell needs.

*Your answer:*

## 4. Judgment cell

The cell below is scored 0 to 2 by the rubric in the syllabus, in the format that `ai-practices.md` gives, and you write it yourself. Part (a): which integration time and which background treatment would you set for your lab, and what MDL would it report? Cite, under the treatment you choose, your spike MDL, blank MDL and the greater at that time, how many of your 40 homes it reports as numbers, and how many calls the spike MDL alone would have changed, and say what the other treatment would have cost; then say what one of those changed calls, or the drift A leaves in, would put in a home's record. Part (b): one judgment entry, a suggestion from the assistant that you changed, rejected or verified, and the reason from your data. The two conversations before TASK 2 are the natural source; the reason has to cite a value from your output, such as your blank MDL against the first answer's, not a general rule. This session's AI practice, named in the notes, is *correct early, and clear context*, and an entry that came from using it is the expected one, though any entry grounded in your data counts.

**Judgment cell**

**(a)**

**(b) Judgment entry**

- **What the assistant suggested:**
- **What I did:**
- **Why, from my data:** 

## Software environment

The last cell prints the Python and package versions this notebook ran with, so a result can always be traced to the software that produced it.

In [ ]:
import matplotlib
import scipy

print(f"Python     {sys.version.split()[0]}")
print(f"NumPy      {np.__version__}")
print(f"pandas     {pd.__version__}")
print(f"SciPy      {scipy.__version__}")
print(f"Matplotlib {matplotlib.__version__}")